# <span style="color: green;"><b>Topic 2: Machine Learning in Big Data</b></span>

### <span style="color: green;"><u>Cleaning Data</u></span>

##### <span style="color: yellow;">Preprocessing is the most critical phase to ensure that Machine Learning models don't just learn noise; I will proceed to prepare this dataset for both supervised and unsupervised tasks. The selected dataset comes from Citibike, New York https://citibikenyc.com/system-data.</span>

### <b>Dataset</b>

In [35]:
import pandas as pd
# Load the dataset *** I used the parameter low_memory because the size of the file
df = pd.read_csv('citibike.csv', low_memory=False)
df.info(verbose=True,max_cols=12)

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             str    
 1   rideable_type       str    
 2   started_at          str    
 3   ended_at            str    
 4   start_station_name  str    
 5   start_station_id    str    
 6   end_station_name    str    
 7   end_station_id      str    
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       str    
dtypes: float64(4), str(9)
memory usage: 99.2 MB


### <b>Handling Nulls</b>
##### <span style="color: yellow;">This dataset has 1,000,001 rows, but there are significant missing values, particularly in the End Station and End Coordinates fields.</span>
##### <span style="color: yellow;">Since there are missing data:</span>

In [36]:
# count of nulls per column
print(df.isnull().sum())
print('='*40)

# percentage of nulls
print((df.isnull().sum() / len(df)) * 100)

ride_id                  0
rideable_type            0
started_at               0
ended_at                 0
start_station_name     903
start_station_id       903
end_station_name      3885
end_station_id        5047
start_lat              903
start_lng              903
end_lat               5003
end_lng               5003
member_casual            0
dtype: int64
ride_id               0.0000
rideable_type         0.0000
started_at            0.0000
ended_at              0.0000
start_station_name    0.0903
start_station_id      0.0903
end_station_name      0.3885
end_station_id        0.5047
start_lat             0.0903
start_lng             0.0903
end_lat               0.5003
end_lng               0.5003
member_casual         0.0000
dtype: float64


### <b>Imputing</b>
##### <span style="color: yellow;">This script creates a dictionary that maps each station name to its average geographic coordinates found elsewhere in the dataset</span>

In [37]:
# build the reference map
# take all rows that have coordinates and group them by station name
reference_data = df.dropna(subset=['end_lat', 'end_lng', 'end_station_name'])
coords_map = reference_data.groupby('end_station_name')[['end_lat', 'end_lng']].mean().to_dict()

# extract individual maps for lat and lng
lat_map = coords_map['end_lat']
lng_map = coords_map['end_lng']

# count missing values
print(f"Nulls in end_lat before: {df['end_lat'].isnull().sum()}")

# apply the Imputation
# It only fills nulls where the 'end_station_name' provides a match in our map
df['end_lat'] = df['end_lat'].fillna(df['end_station_name'].map(lat_map))
df['end_lng'] = df['end_lng'].fillna(df['end_station_name'].map(lng_map))

# Checking the improvement
print(f"Nulls in end_lat after: {df['end_lat'].isnull().sum()}")

Nulls in end_lat before: 5003
Nulls in end_lat after: 3841


##### <span style="color: yellow;">we can use the end_station_id to fill missing end_station_names, and vice versa</span>

In [38]:
# Now build the Knowledge Base
# select rows where both name and Id exist to create our mapping dictionaries
ref_df = df.dropna(subset=['end_station_name', 'end_station_id'])

# create a map to find the Name if I have the ID
id_to_name_map = ref_df.drop_duplicates('end_station_id').set_index('end_station_id')['end_station_name'].to_dict()

# Create a map to find the ID if I have the Name
name_to_id_map = ref_df.drop_duplicates('end_station_name').set_index('end_station_name')['end_station_id'].to_dict()

# apply the Imputation
# Fill missing Names by looking up the Id
df['end_station_name'] = df['end_station_name'].fillna(df['end_station_id'].map(id_to_name_map))

# Fill missing Ids by looking up the Name
df['end_station_id'] = df['end_station_id'].fillna(df['end_station_name'].map(name_to_id_map))

# Check results
print(f"Remaining null names: {df['end_station_name'].isnull().sum()}")
print(f"Remaining null IDs: {df['end_station_id'].isnull().sum()}")

Remaining null names: 3885
Remaining null IDs: 3885


##### <span style="color: yellow;">After applying the Lookup Table, the missing data was reduced to:</span>

In [39]:
# modified dataset
print((df.isnull().sum() / len(df)) * 100)

ride_id               0.0000
rideable_type         0.0000
started_at            0.0000
ended_at              0.0000
start_station_name    0.0903
start_station_id      0.0903
end_station_name      0.3885
end_station_id        0.3885
start_lat             0.0903
start_lng             0.0903
end_lat               0.3841
end_lng               0.3841
member_casual         0.0000
dtype: float64


##### <span style="color: yellow;">Drop any row with at least one null</span>

In [40]:
# new cleaned dataset
df_cleaned = df.dropna()
print((df_cleaned.isnull().sum() / len(df)) * 100)


ride_id               0.0
rideable_type         0.0
started_at            0.0
ended_at              0.0
start_station_name    0.0
start_station_id      0.0
end_station_name      0.0
end_station_id        0.0
start_lat             0.0
start_lng             0.0
end_lat               0.0
end_lng               0.0
member_casual         0.0
dtype: float64


### <b>Filter Ghost Trips</b>
##### <span style="color: yellow;">Any trip with a duration <= 0 or trips where the start and end coordinates are exactly the same with a duration of less than a minute (impossible duration or immediate cancellation) should be removed, as these often represent system tests or failed unlocks.</span>
##### <span style="color: yellow;">After applying the filtering, I got zero ghost trips; it is very likely that those cases were removed in the drop rows previous step.</span>

##### <span style="color: yellow;">This process involves converting the timestamp strings into datetime objects to perform arithmetic, then creating a mask to remove the noise</span>

In [41]:
# convert to datetime to calculate duration
df_cleaned['started_at'] = pd.to_datetime(df_cleaned['started_at'])
df_cleaned['ended_at'] = pd.to_datetime(df_cleaned['ended_at'])

# calculate trip duration in seconds
df_cleaned['duration_sec'] = (df_cleaned['ended_at'] - df_cleaned['started_at']).dt.total_seconds()

# define the filtering logic
# rule 1: Must have a duration greater than zero
# rule 2: If the start and end stations are the same,
# it must last longer than 60 seconds to be considered a real trip
is_ghost_trip = (
    (df_cleaned['duration_sec'] <= 0) |
    ((df_cleaned['start_station_id'] == df_cleaned['end_station_id']) & (df_cleaned['duration_sec'] < 60))
)

# apply the filter
# keep only the rows that are not ghost trips (using symbol ~)
df_cleaned_new = df_cleaned[~is_ghost_trip].copy()

# report the findings
removed_count = len(df_cleaned) - len(df_cleaned_new)
print(f"Removed {removed_count} Ghost Trips from the dataset.")


Removed 0 Ghost Trips from the dataset.


### <b>Deduplication</b>
##### <span style="color: yellow;">process of identifying and removing redundant records</span>

In [42]:
duplicate_count = df_cleaned.duplicated(subset=['ride_id']).sum()
print(f"Number of duplicate ride_ids: {duplicate_count}")

if duplicate_count > 0:
    print(df_cleaned[df_cleaned.duplicated(subset=['ride_id'], keep=False)].sort_values('ride_id').head())

Number of duplicate ride_ids: 0


##### <span style="color: yellow;">sometimes, two records have different ride_ids but represent the same real-world event (Logical duplicates)</span>

In [43]:
logical_dupes = df_cleaned.duplicated(subset=['started_at', 'start_station_id', 'rideable_type'], keep=False)

print(f"Potential logical duplicates: {logical_dupes.sum()}")

Potential logical duplicates: 0


### <b>Feature Transformation</b>
##### <span style="color: yellow;">Raw timestamps and strings are not directly useful for most algorithms. We must transform them into numerical signals</span>
##### <span style="color: yellow;">We have to treat the timestamp strings as actual datetime objects</span>

In [44]:
import numpy as np
# convert strings to datetime objects
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])


### <b>Calculating Trip Duration</b>
##### <span style="color: yellow;">the length of a trip is often the strongest indicator of user behavior; we have to treat the timestamp strings as actual datetime objects</span>

In [45]:
# calculate duration in minutes
df['duration_min'] = (df['ended_at'] - df['started_at']).dt.total_seconds() / 60

# checking the first few values
print(df[['started_at', 'ended_at', 'duration_min']].head())

               started_at                ended_at  duration_min
0 2026-02-06 17:20:08.324 2026-02-06 17:24:21.584      4.221000
1 2026-02-04 12:02:28.817 2026-02-04 12:10:27.340      7.975383
2 2026-02-11 19:37:21.253 2026-02-11 19:43:18.957      5.961733
3 2026-02-01 12:31:54.323 2026-02-01 12:39:24.020      7.494950
4 2026-02-13 05:49:37.127 2026-02-13 06:09:27.904     19.846283


##### <span style="color: yellow;">We break the timestamp into simple integers. This helps the model identify patterns like "morning rushes" or "weekend peaks"</span>

In [46]:
# extract hour (0-23)
df['hour'] = df['started_at'].dt.hour

# extract day of week [ 0 = monday ]
df['day_of_week'] = df['started_at'].dt.dayofweek

# extract day of month
df['day_of_month'] = df['started_at'].dt.day

# checking the first few values
print(df[['hour', 'day_of_week', 'day_of_month']].head())

   hour  day_of_week  day_of_month
0    17            4             6
1    12            2             4
2    19            2            11
3    12            6             1
4     5            4            13


### <b>Creating Binary Indicators</b>
##### <span style="color: yellow;">create specific "flags" for known business patterns like weekends and peak commuting hours</span>

In [47]:
# is it a weekend? (5 = Saturday or 6 = Sunday)
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# is it peak commuting hours? (for instance 7-9 and 14-19)
df['is_rush_hour'] = df['hour'].isin([7, 8, 9, 16, 17, 18, 19]).astype(int)

# checking the first few values
print(df[['is_weekend', 'is_rush_hour']].head())

   is_weekend  is_rush_hour
0           0             1
1           0             0
2           0             1
3           1             0
4           0             0

### <b>Cyclical Time</b>
##### <span style="color: yellow;">The goal is to extract hour of day (0-23) and day of week</span>
##### <span style="color: yellow;">To help the model understand that hour 23 is close to hour 0, we will use sine/cosine transformations</span>

In [48]:
# 24 hours in a day
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
# Now, hour 23 and hour 0 will be mathematically close in the vector space
# checking the first few values
print(df[['hour_sin','hour_cos']].head())

       hour_sin  hour_cos
0 -9.659258e-01 -0.258819
1  1.224647e-16 -1.000000
2 -9.659258e-01  0.258819
3  1.224647e-16 -1.000000
4  9.659258e-01  0.258819


### <b>Categorical Encoding</b>
##### <span style="color: yellow;">It is the process of translating labels into a numerical format that preserves the information's meaning</span>
##### <span style="color: yellow;">The task is to convert rideable_type (Electric vs. Classic) and member_casual into 0 and 1</span>

In [49]:
# encode member_casual [ member = 1, casual = 0 ]
df['is_member'] = df['member_casual'].map({'member': 1, 'casual': 0})

# encode 'rideable_type' [ electric_bike = 1, classic_bike = 0 ]
df['is_electric'] = df['rideable_type'].map({'electric_bike': 1, 'classic_bike': 0})

print(df[['is_member', 'is_electric']].head())

   is_member  is_electric
0          1            1
1          1            1
2          1            1
3          1            1
4          1            0


### <b>Frequency Encoding</b>
##### <span style="color: yellow;">For Supervised tasks, we must avoid one-hot encoding (which creates a sparse matrix); instead, we will use Frequency Encoding and  Label Encoding to represent stations in a lower-dimensional space</span>
##### <span style="color: yellow;">This replaces the station name with the number of times it appears in the dataset. This captures the "popularity" of a station as a numerical signal</span>

In [50]:
# frequency of each start station
station_freq = df['start_station_name'].value_counts()

# mapping the frequencies back to the dataframe
df['start_station_popularity'] = df['start_station_name'].map(station_freq)

print(df[['start_station_name', 'start_station_popularity']].head())

        start_station_name  start_station_popularity
0      Bond St & Bergen St                     479.0
1  Norman Ave & Leonard St                     857.0
2           E 2 St & Ave C                    1527.0
3  St James Pl & Oliver St                    1284.0
4          W 20 St & 7 Ave                    2002.0


### <b>Normalization and Scaling</b>
##### <span style="color: yellow;">I decided to use Standardization since we have some very long trips (outliers), Min-Max scaling would compress almost all our data into the range of 0.0 to 0.05, while Standardization will keep the data spread out and meaningful for the model</span>
##### <span style="color: yellow;">Since start_lat, start_lng, and duration have different units, we are going to apply Standardization to give them a mean of 0 and a standard deviation of 1</span>

In [51]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

df['duration_min'] = (pd.to_datetime(df['ended_at']) - pd.to_datetime(df['started_at'])).dt.total_seconds() / 60

# columns to scale
cols_to_scale = ['duration_min', 'start_lat', 'start_lng']

# applying Standardization (Z-score)
scaler_std = StandardScaler()
df[cols_to_scale] = scaler_std.fit_transform(df[cols_to_scale])

# Now your duration, latitude, and longitude are all on a
# similar mathematical footing (centered around 0)
print(df[cols_to_scale].describe())

       duration_min     start_lat     start_lng
count  1.000000e+06  9.990970e+05  9.990970e+05
mean   1.818989e-18  6.948265e-14 -2.939959e-14
std    1.000001e+00  1.000001e+00  1.000001e+00
min   -2.932891e-01 -3.206422e+00 -2.457196e+00
25%   -1.823454e-01 -5.911288e-01 -7.046120e-01
50%   -9.860970e-02 -5.074383e-03 -2.625713e-01
75%    3.996420e-02  5.865997e-01  5.871792e-01
max    4.097339e+01  3.840520e+00  4.519964e+00


### <b>Addressing Data Imbalance</b>
##### <span style="color: yellow;">Initial inspection shows a significant imbalance in the member_casual column:</span>


In [52]:
# getting the Raw Counts
counts = df['member_casual'].value_counts()

# get the percentages ->> Relative Frequency
# setting normalize=True gives us the proportion (0.0 to 1.0)
percentages = df['member_casual'].value_counts(normalize=True) * 100

# the results
print("== Class Distribution ==")
print(counts)
print("\n== Percentage Distribution ==")
print(percentages.map('{:.2f}%'.format))

== Class Distribution ==
member_casual
member    914503
casual     85497
Name: count, dtype: int64

== Percentage Distribution ==
member_casual
member    91.45%
casual     8.55%
Name: proportion, dtype: str


### <span style="color: green;"><u>Supervised Learning</u></span>
##### <span style="color: yellow;">This is a dataset of 1,000,000 rows, so we need an algorithm that is computationally efficient, handles non-linear patterns (like traffic flow), and is robust against the noise typical of Big Data</span>

### <b>Random Forest algorithm</b>
##### <span style="color: yellow;">I have selected the Random Forest Classifier for this task. Taking in account that we are handeling 1,000,000-row scale the reasons for using this algorithm are: Parallel Processing, Handling Non-Linearity, Robustness to Outliers, Feature Importance, Implementation</span>

In [53]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler

# prepare features (X) and target (y)
X = df[['duration_min', 'hour_sin', 'hour_cos', 'start_lat', 'start_lng', 'is_electric']]
y = df['is_member'] # 1 for member, 0 for casual

# first: train-Test split
# We keep 20% of the 1M rows for the final, honest evaluation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# second: under-sampling (Only on Training Data)
# This creates a balanced 50/50 dataset for the model to learn from
rus = RandomUnderSampler(random_state=42)
X_train_bal, y_train_bal = rus.fit_resample(X_train, y_train)

# third: train the random forest
# n_jobs=-1 tells the computer to use all available CPU cores for Big Data speed
model = RandomForestClassifier(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42)
model.fit(X_train_bal, y_train_bal)

# fourth: predict on the UNBALANCED test set
y_pred = model.predict(X_test)

### <b>Evaluating Model Performance</b>
##### <span style="color: yellow;">In Big Data, Accuracy is a trap. If 91.45% of our riders are members, a broken model that always says Member is 91.45% accurate but 0% useful. We must use a Confusion Matrix and better metrics:</span>

In [54]:
# Print the results
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.13      0.64      0.22     17041
           1       0.95      0.61      0.75    182959

    accuracy                           0.62    200000
   macro avg       0.54      0.63      0.48    200000
weighted avg       0.88      0.62      0.70    200000

--- Confusion Matrix ---
[[ 10989   6052]
 [ 70711 112248]]


### <span style="color: green;"><u>Testing Model</u>
##### <span style="color: yellow;">With 1,000,000 rows, Random Forest can now predict with high precision whether a trip is a Casual or Member based purely on the trip's duration and start time. This is Business Intelligence at scale. We can now target specific stations at specific hours with Membership Upgrade promotions, knowing exactly who the casual riders are</span>


In [ ]:
# ============================================================
# RANDOM TRIP PREDICTION — Supervised Model Evaluation
# ============================================================

# 1 to 90 minutes
raw_duration_min = np.random.uniform(1, 90)
# 0 to 23
random_hour = np.random.randint(0, 24)
# NYC Citi Bike lat range
raw_start_lat = np.random.uniform(40.65, 40.80)
# NYC Citi Bike lng range
raw_start_lng = np.random.uniform(-74.05, -73.90)
# 0=classic, 1=electric
is_electric = np.random.choice([0, 1])

# scale duration, lat, lng with the same scaler_std from the notebook
# scaler_std was fitted on ['duration_min', 'start_lat', 'start_lng']
raw_input = pd.DataFrame(
    [[raw_duration_min, raw_start_lat, raw_start_lng]],
    columns=['duration_min', 'start_lat', 'start_lng']
)
scaled_vals = scaler_std.transform(raw_input)
duration_scaled = scaled_vals[0, 0]
lat_scaled = scaled_vals[0, 1]
lng_scaled = scaled_vals[0, 2]

# cyclical encoding of hour
hour_sin = np.sin(2 * np.pi * random_hour / 24)
hour_cos = np.cos(2 * np.pi * random_hour / 24)

# build feature vector (same column order as training)
X_new = pd.DataFrame(
    [[duration_scaled, hour_sin, hour_cos, lat_scaled, lng_scaled,
 is_electric]],
    columns=['duration_min', 'hour_sin', 'hour_cos', 'start_lat',
 'start_lng', 'is_electric']
)

# predict
prediction    = model.predict(X_new)[0]
# [prob_casual, prob_member]
probabilities = model.predict_proba(X_new)[0]
label = "MEMBER  (annual subscriber)" if prediction == 1 else  "CASUAL (pay-per-ride)"
bike_type = "Electric bike" if is_electric else "Classic bike"
rush = "Rush hour" if random_hour in [7, 8, 9, 16, 17, 18, 19] else "Off-peak"

# display
print("=" * 58)
print("       RANDOM TRIP  —  RANDOM FOREST PREDICTION")
print("=" * 58)
print(f"  Duration    : {raw_duration_min:.1f} minutes")
print(f"  Start hour  : {random_hour:02d}:00  ({rush})")
print(f"  Start lat   : {raw_start_lat:.5f}")
print(f"  Start lng   : {raw_start_lng:.5f}")
print(f"  Bike type   : {bike_type}  (is_electric={is_electric})")
print("-" * 58)
print(f"  PREDICTION  : {label}")
print(f"  Confidence  : Member={probabilities[1]:.1%} | Casual={probabilities[0]:.1%}")
print("=" * 58)



       RANDOM TRIP  —  RANDOM FOREST PREDICTION
  Duration    : 28.9 minutes
  Start hour  : 03:00  (Off-peak)
  Start lat   : 40.77037
  Start lng   : -73.90654
  Bike type   : Electric bike  (is_electric=1)
----------------------------------------------------------
  PREDICTION  : MEMBER  (annual subscriber)
  Confidence  : Member=55.1% | Casual=44.9%
